In [ ]:
cell = "GM12878"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
# Update this root_dir to your own project path before running
root_dir = "/mnt/d/TAD-MultiOmicsNet"
DS_TAG = ["10r1", "10r2", "10r3", "25r1", "25r2", "25r3", "50r1", "50r2", "50r3", "75r1", "75r2", "75r3"]

In [ ]:
### chr1-12 Generation of training samples for all positive and negative samples

In [ ]:
import os
import numpy as np

def openreadtxt(file_name):
    with open(file_name, 'r') as file:
        file_data = file.readlines()
        data = [row.replace('\n', '').split() for row in file_data]
    return data

def read_hic_matrix_chunk(file_path, chunk_size=1000):
    # Read the Hi-C matrix text in blocks line by line
    with open(file_path, 'r') as f:
        while True:
            chunk_lines = f.readlines(chunk_size * 10240)
            if not chunk_lines:
                break
            for line in chunk_lines:
                line = line.strip()
                if not line:
                    continue
                row_arr = np.array(line.split(), dtype=np.float32)
                yield row_arr

if __name__ == '__main__':
    chrs = list(range(1, 13))
    pad_up, pad_down = 4, 5
    pad_left, pad_right = 4, 5
    win_half = 5

    for ds_tag in DS_TAG:
        print(f"=========================={ds_tag}====================")
        for chr in chrs:
            all_data = []
            label_T = 1
            label_F = 0
            ww = 0
    
            boundary_rows = openreadtxt(f"{root_dir}/{cell}/{display_reso}kb/boundary-generate/chr{chr}_{cell}_{display_reso}kb_boundary.txt")
            n_bound = len(boundary_rows)
    
            matrix_rows = []
            print(f"Start chunked reading chr{chr} Hi-C matrix")
            for row in read_hic_matrix_chunk(f'{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/prepare_data/{cell}_{display_reso}kb_{ds_tag}.chr{chr}'):
                pad_row = np.pad(row, (pad_left, pad_right), mode='constant')
                matrix_rows.append(pad_row)
            big_len = len(matrix_rows)
            row_dim = matrix_rows[0].shape[0]
    
            # Add zero padding to top and bottom of matrix
            top_pad = np.zeros((pad_up, row_dim), dtype=np.float32)
            bottom_pad = np.zeros((pad_down, row_dim), dtype=np.float32)
            padded_matrix = np.vstack([top_pad, np.vstack(matrix_rows), bottom_pad])
            print(f"Number of rows in the original matrix: {big_len}, matrix shape after padding: {padded_matrix.shape}")
    
            # m starts from 5
            for m in range(5, 5 + big_len):
                if ww < n_bound:
                    bp_pos = int(boundary_rows[ww][0])
                    bin_size = resolution
                    num_key = bp_pos // bin_size
                    if m == num_key + 5:
                        current_label = label_T
                        ww += 1
                    else:
                        current_label = label_F
                else:
                    current_label = label_F
    
                midx = m
                sub_mat = padded_matrix[midx - win_half : midx + win_half, midx - win_half : midx + win_half]
                sample = np.append(sub_mat.flatten(), current_label)
                all_data.append(sample)
    
            all_data = np.array(all_data, dtype='float32')
            print(f"chr{chr}, shape of the final sample set: {all_data.shape}")
            
            tra_npy = f"{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/samples-generate/train"
            os.makedirs(tra_npy, exist_ok=True)
            
            np.save(f'{tra_npy}/{cell}_{display_reso}kb_chr{chr}_{ds_tag}.npy', all_data)
            print(f"chr{chr}, saved successfully\n")



In [ ]:
### Negative sample sampling: maintain 1:4 positive-to-negative sample ratio
# Randomly sample negative samples without replacement

In [ ]:
import numpy as np
import pandas as pd
import os

SEED_LIST = [42, 43, 44, 45, 46] 
bin_size = resolution
chrs = list(range(1, 13))

for ds_tag in DS_TAG:
    print(f"=========================={ds_tag}====================")
    root_path = f'{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/samples-generate/train'
    
    for SEED in SEED_LIST:
        print(f"====== Generate dataset, SEED = {SEED} ======")
        np.random.seed(SEED) 
        all_hic_samples = []
        all_coords = []
        
        for chr_id in chrs:
            print(f"chr{chr_id}")
            file_path = f'{root_path}/{cell}_{display_reso}kb_chr{chr_id}_{ds_tag}.npy'
            if not os.path.exists(file_path):
                print(f"Warning：{file_path} is not found，skip chr{chr_id}")
                continue
                
            train_label = np.load(file_path)
            total_bin = len(train_label)
        
            bin_coords = []
            for idx in range(total_bin):
                s = idx * bin_size
                e = s + bin_size
                bin_coords.append([f"chr{chr_id}", s, e])
        
            # Positive sample screening
            pos_mask = train_label[:, -1] == 1
            pos_idx = np.nonzero(pos_mask)[0]
            if len(pos_idx) == 0:
                print(f"chr{chr_id} has no positive samples, skip")
                continue
        
            for idx in pos_idx:
                all_hic_samples.append(train_label[idx:idx+1])
                all_coords.append(bin_coords[idx])
        
            # Negative sample sampling: 4 negatives per positive sample
            neg_idx = np.nonzero(~pos_mask)[0]
            need_neg = len(pos_idx) * 4
            select_num = min(need_neg, len(neg_idx))
            select_neg = np.random.choice(neg_idx, size=select_num, replace=False)
            
            for idx in select_neg:
                all_hic_samples.append(train_label[idx:idx+1])
                all_coords.append(bin_coords[idx])
        
        if len(all_hic_samples) == 0:
            raise RuntimeError("No positive or negative samples were collected. Please check the labels in the input npy file.")   
        hic_train = np.concatenate(all_hic_samples, axis=0)
        assert hic_train.shape[0] == len(all_coords), "The number of rows of the sample matrix does not match that of the coordinates!"
        
        np.save(f'{root_path}/{cell}_{display_reso}kb_train_{ds_tag}_seed{SEED}.npy', hic_train)
        print(f"Hi-C raw training set saved, total shape：{hic_train.shape}")
        
        coord_df = pd.DataFrame(all_coords, columns=["chr", "start", "end"])
        coord_df.to_csv(f'{root_path}/hic_train_{ds_tag}_{display_reso}kb_seed{SEED}.csv', index=False)
        

In [ ]:
## Chr13-19 validation set generation

In [ ]:
import os
import numpy as np

def openreadtxt(file_name):
    with open(file_name, 'r') as file:
        file_data = file.readlines()
        data = [row.replace('\n', '').split() for row in file_data]
    return data

def read_hic_matrix_chunk(file_path, chunk_size=1000):
    with open(file_path, 'r') as f:
        while True:
            chunk_lines = f.readlines(chunk_size * 10240)
            if not chunk_lines:
                break
            for line in chunk_lines:
                line = line.strip()
                if not line:
                    continue
                row_arr = np.array(line.split(), dtype=np.float32)
                yield row_arr

if __name__ == '__main__':
    chrs=list(range(13,20))
    chrs = [str(x) for x in chrs]
    pad_up, pad_down = 4, 5
    pad_left, pad_right = 4, 5
    win_half = 5
    
    for ds_tag in DS_TAG:
        print(f"=========================={ds_tag}====================")
        for chr in chrs:
            all_data = []
            label_T = 1
            label_F = 0
            ww = 0
    
            boundary_rows = openreadtxt(f'{root_dir}/{cell}/{display_reso}kb/boundary-generate/chr{chr}_{cell}_{display_reso}kb_boundary.txt')
            n_bound = len(boundary_rows)
    
            matrix_rows = []
            print(f"Start chunked reading chr{chr} Hi-C matrix")
            for row in read_hic_matrix_chunk(f'{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/prepare_data/{cell}_{display_reso}kb_{ds_tag}.chr{chr}'):
                pad_row = np.pad(row, (pad_left, pad_right), mode='constant')
                matrix_rows.append(pad_row)
            big_len = len(matrix_rows)
            row_dim = matrix_rows[0].shape[0]
    
            # Add zero padding to top and bottom of matrix
            top_pad = np.zeros((pad_up, row_dim), dtype=np.float32)
            bottom_pad = np.zeros((pad_down, row_dim), dtype=np.float32)
            padded_matrix = np.vstack([top_pad, np.vstack(matrix_rows), bottom_pad])
            print(f"Number of rows in the original matrix: {big_len}, matrix shape after padding: {padded_matrix.shape}")
    
            # m starts from 5
            for m in range(5, 5 + big_len):
                if ww < n_bound:
                    bp_pos = int(boundary_rows[ww][0])
                    bin_size = resolution
                    num_key = bp_pos // bin_size
                    if m == num_key + 5:
                        current_label = label_T
                        ww += 1
                    else:
                        current_label = label_F
                else:
                    current_label = label_F
    
                midx = m
                sub_mat = padded_matrix[midx - win_half : midx + win_half, midx - win_half : midx + win_half]
                sample = np.append(sub_mat.flatten(), current_label)
                all_data.append(sample)
    
            all_data = np.array(all_data, dtype='float32')
            print(f"chr{chr}, shape of the final sample set: {all_data.shape}")
            
            val_npy = f"{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/samples-generate/validation"
            os.makedirs(val_npy, exist_ok=True)
            
            np.save(f'{val_npy}/{cell}_{display_reso}kb_chr{chr}_{ds_tag}.npy', all_data)
            print(f"chr{chr}, saved successfully\n")

In [ ]:
### chr20-23 prediction set generation

In [ ]:
import os
import numpy as np

def openreadtxt(file_name):
    with open(file_name, 'r') as file:
        file_data = file.readlines()
        data = [row.replace('\n', '').split() for row in file_data]
    return data

def read_hic_matrix_chunk(file_path, chunk_size=1000):
    with open(file_path, 'r') as f:
        while True:
            chunk_lines = f.readlines(chunk_size * 10240)
            if not chunk_lines:
                break
            for line in chunk_lines:
                line = line.strip()
                if not line:
                    continue
                row_arr = np.array(line.split(), dtype=np.float32)
                yield row_arr

if __name__ == '__main__':
    chrs=list(range(20,23))
    chrs = [str(x) for x in chrs]
    chrs.append('X')
    pad_up, pad_down = 4, 5
    pad_left, pad_right = 4, 5
    win_half = 5
    
    for ds_tag in DS_TAG:
        print(f"=========================={ds_tag}====================")
        for chr in chrs:
            all_data = []
            label_T = 1
            label_F = 0
            ww = 0
    
            boundary_rows = openreadtxt(f'{root_dir}/{cell}/{display_reso}kb/boundary-generate/chr{chr}_{cell}_{display_reso}kb_boundary.txt')
            n_bound = len(boundary_rows)
    
            matrix_rows = []
            print(f"Start chunked reading chr{chr} Hi-C matrix")
            for row in read_hic_matrix_chunk(f'{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/prepare_data/{cell}_{display_reso}kb_{ds_tag}.chr{chr}'):
                pad_row = np.pad(row, (pad_left, pad_right), mode='constant')
                matrix_rows.append(pad_row)
            big_len = len(matrix_rows)
            row_dim = matrix_rows[0].shape[0]
    
            # Add zero padding to top and bottom of matrix
            top_pad = np.zeros((pad_up, row_dim), dtype=np.float32)
            bottom_pad = np.zeros((pad_down, row_dim), dtype=np.float32)
            padded_matrix = np.vstack([top_pad, np.vstack(matrix_rows), bottom_pad])
            print(f"Number of rows in the original matrix: {big_len}, matrix shape after padding: {padded_matrix.shape}")
    
            # m starts from 5
            for m in range(5, 5 + big_len):
                if ww < n_bound:
                    bp_pos = int(boundary_rows[ww][0])
                    bin_size = resolution
                    num_key = bp_pos // bin_size
                    if m == num_key + 5:
                        current_label = label_T
                        ww += 1
                    else:
                        current_label = label_F
                else:
                    current_label = label_F
    
                midx = m
                sub_mat = padded_matrix[midx - win_half : midx + win_half, midx - win_half : midx + win_half]
                sample = np.append(sub_mat.flatten(), current_label)
                all_data.append(sample)
    
            all_data = np.array(all_data, dtype='float32')
            print(f"chr{chr}, shape of the final sample set: {all_data.shape}")
            
            pre_npy = f"{root_dir}/{cell}/sample/{display_reso}kb/{ds_tag}/samples-generate/predict"
            os.makedirs(pre_npy, exist_ok=True)
            
            np.save(f'{pre_npy}/{cell}_{display_reso}kb_chr{chr}_{ds_tag}.npy', all_data)
            print(f"chr{chr}, saved successfully\n")
        